# Connect Automation Walkthrough

This notebook demonstrates establishing a stubbed Connect session, composing the Open control using the SDK helpers, and replaying a wallet approval frame without requiring a running Torii node.

## Flow Overview

1. Simulate the Torii Connect endpoints with an in-memory session.
2. Create a Connect session and inspect the advertised policy limits.
3. Build an Open frame with the CLI metadata helpers and submit it to the stub.
4. Recreate an approval frame and decode it to verify the payload contents.

In [ ]:
from __future__ import annotations

import json
from typing import Any, Dict, Optional
from urllib.parse import urlparse
from importlib import resources

from iroha_python import (
    ConnectAppMetadata,
    ConnectControlApprove,
    ConnectPermissions,
    ConnectDirection,
    ConnectFrame,
    generate_connect_keypair,
    encode_connect_frame,
    decode_connect_frame,
)
from iroha_python.client import ToriiClient
from iroha_python.examples.connect_flow import build_connect_open_frame


class NotebookResponse:
    """Minimal response stub that mimics :class:`requests.Response`."""

    def __init__(self, payload: Optional[Dict[str, Any]], status_code: int = 200) -> None:
        self.status_code = status_code
        self._payload = payload
        self.headers: Dict[str, str] = {"Content-Type": "application/json"}
        self._content = (
            json.dumps(payload).encode("utf-8") if payload is not None else b""
        )
        self.content = self._content
        self.text = self._content.decode("utf-8") if payload is not None else ""

    def json(self) -> Dict[str, Any]:
        if self._payload is None:
            raise ValueError("no JSON payload available")
        return json.loads(self.text or "null")

    def close(self) -> None:
        return None


class NotebookSession:
    """In-memory `requests.Session` replacement for notebook automation."""

    def __init__(self) -> None:
        self.calls: list[Dict[str, Any]] = []
        self.sid = "0x000102030405060708090a0b0c0d0e0f101112131415161718191a1b1c1d1e1f"
        self.status_snapshot: Dict[str, Any] = {
            "enabled": True,
            "sessions_total": 1,
            "sessions_active": 0,
            "per_ip_sessions": [{"ip": "127.0.0.1", "sessions": 0}],
            "buffered_sessions": 0,
            "total_buffer_bytes": 0,
            "dedupe_size": 0,
            "policy": {
                "ws_max_sessions": 32,
                "ws_per_ip_max_sessions": 4,
                "ws_rate_per_ip_per_min": 60,
                "session_ttl_ms": 600_000,
                "frame_max_bytes": 65_536,
                "session_buffer_max_bytes": 262_144,
                "relay_enabled": True,
            },
            "frames_in_total": 0,
            "frames_out_total": 0,
            "ciphertext_total": 0,
            "dedupe_drops_total": 0,
            "buffer_drops_total": 0,
            "plaintext_control_drops_total": 0,
            "monotonic_drops_total": 0,
        }

    def request(
        self,
        method: str,
        url: str,
        params: Optional[Dict[str, Any]] = None,
        headers: Optional[Dict[str, str]] = None,
        data: Optional[bytes] = None,
        stream: bool = False,
        timeout: Optional[float] = None,
    ) -> NotebookResponse:
        parsed = urlparse(url)
        path = parsed.path
        body: Any = None
        if data:
            try:
                body = json.loads(data.decode("utf-8"))
            except Exception:
                body = data.decode("utf-8")

        verb = method.upper()
        if verb == "POST" and path == "/v1/connect/session":
            payload: Dict[str, Any] = {
                "sid": self.sid,
                "app_uri": f"iroha://connect?sid={self.sid}",
                "token_app": "app-token-demo",
                "token_wallet": "wallet-token-demo",
            }
        elif verb == "GET" and path == "/v1/connect/status":
            payload = self.status_snapshot
        elif verb == "POST" and path.startswith("/v1/connect/control/"):
            payload = {"accepted": True, "endpoint": path}
        else:
            payload = {}

        self.calls.append({
            "method": verb,
            "path": path,
            "body": body,
            "headers": dict(headers or {}),
        })
        return NotebookResponse(payload)

    def get(
        self,
        url: str,
        *,
        params: Optional[Dict[str, Any]] = None,
        stream: bool = False,
        timeout: Optional[float] = None,
        headers: Optional[Dict[str, str]] = None,
    ) -> NotebookResponse:
        return self.request(
            "GET",
            url,
            params=params,
            headers=headers,
            data=None,
            stream=stream,
            timeout=timeout,
        )


session = NotebookSession()
client = ToriiClient("http://127.0.0.1:8080", session=session)


In [ ]:
session_info = client.create_connect_session_info({"role": "app"})
session_info


In [ ]:
status_snapshot = client.get_connect_status_typed()
{"sessions_total": status_snapshot.sessions_total, "session_ttl_ms": status_snapshot.policy.session_ttl_ms if status_snapshot.policy else None}


In [ ]:
template_text = resources.files("iroha_python.examples").joinpath("connect_app_metadata.json").read_text(encoding="utf-8")
metadata_dict = json.loads(template_text)
metadata_dict["name"] = "Notebook Demo dApp"
metadata = ConnectAppMetadata.from_dict(metadata_dict)
metadata.to_dict()


In [ ]:
app_keys = generate_connect_keypair()
open_frame = build_connect_open_frame(
    session_info,
    app_public_key=app_keys.public_key,
    chain_id="dev-chain",
    methods=["SIGN_REQUEST_TX"],
    events=["DISPLAY"],
    metadata=metadata,
)
client.send_connect_control_frame(session_info.sid, open_frame.control)


In [ ]:
wallet_keys = generate_connect_keypair()
approve = ConnectControlApprove(
    wallet_public_key=wallet_keys.public_key,
    account_id="wallet@test",
    signature=b"\x01" * 64,
    permissions=ConnectPermissions(methods=["SIGN_RESULT_OK"], events=[]),
)
approve_frame = ConnectFrame(
    sid=open_frame.sid,
    direction=ConnectDirection.WALLET_TO_APP,
    sequence=open_frame.sequence,
    control=approve,
)
encoded_frame = encode_connect_frame(approve_frame)
decoded_frame = decode_connect_frame(encoded_frame)
decoded_frame.control.to_dict()


In [ ]:
session.calls


## Call Log

The captured `session.calls` list shows each REST interaction the stub recorded during the walkthrough.
